# MJO Hovmöller diagrams and phase composites

This notebook consolidates the legacy precipitation and OLR notebooks. Set `VARIABLE` to `"PRECT"` or `"FLUT"`; generated data and figures use the shared diagnostic output structure.


In [ ]:
# =============================================================================
# TOP-LEVEL PATH PARAMETERS
# Edit these three paths to relocate inputs, the analysis checkout, or outputs.
# =============================================================================
from pathlib import Path
import sys

INPUT_DATA_ROOT = Path("/compyfs/zhan391/v3_dart_cda_scratch")
WORK_DIR = Path("/compyfs/zhan391/work_package/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_dart_2026")
WORKFLOW_NAME = "analysis_mjo"

In [ ]:
# Derived paths: built from the parameters above.
import sys
from pathlib import Path

DIAGNOSTIC_DIR = WORK_DIR / "diagnostic"
if not DIAGNOSTIC_DIR.is_dir():
    raise FileNotFoundError(f"Diagnostic work directory not found: {DIAGNOSTIC_DIR}")
if str(DIAGNOSTIC_DIR) not in sys.path:
    sys.path.insert(0, str(DIAGNOSTIC_DIR))

from configs.output_paths import workflow_output_dirs
WORKFLOW_DATA_DIR, WORKFLOW_FIGURE_DIR = workflow_output_dirs(
    WORKFLOW_NAME,
    output_root=DIAGNOSTIC_OUTPUT_ROOT,
)

## Experiment configuration


In [ ]:
# Select PRECT or FLUT. These settings replace the two legacy notebooks.
VARIABLE = "PRECT"
TIME_START = "20111201"
TIME_END = "20120229"
REGION_BOUNDS = (-10, 10)
MJO_FILTER = VARIABLE == "FLUT"
REMOVE_CLIMATOLOGY = True
MODEL_EXPERIMENTS = {
    "CTRLEN10": (
        "CTRLEN10_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy",
        "CTRLEN10_15day_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy",
        10,
    ),
    "CAPTEN10": (
        "CTRLEN10_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy",
        "CAPTEN10_15day_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy",
        10,
    ),
    "DARTEN20": (
        "DARTEN20_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy",
        "DARTEN20_15day_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy",
        20,
    ),
    "DARTEN40": (
        "DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy",
        "DARTEN40_15day_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy",
        40,
    ),
}


## Build input catalog


In [ ]:
from util.mjo_precipitation_diagnostics import HovmollerPrecipAnalyzer

VARIABLE_INFO = {
    "PRECT": {"alias": "Precipitation", "units": "mm/day", "scale": 86400000.0},
    "FLUT": {"alias": "Outgoing longwave radiation", "units": "W/m2", "scale": 1.0},
}
if VARIABLE not in VARIABLE_INFO:
    raise ValueError(f"VARIABLE must be one of {tuple(VARIABLE_INFO)}")

def segment(run_name, year, nens):
    return {
        "path": str(INPUT_DATA_ROOT / run_name / "archive/post/atm/180x360_aave/ts/daily"),
        "template": f"{VARIABLE}.%(ensemble)s.{year}.nc",
        "period": "20111201-20111231" if year == 2011 else "20120101-20120229",
        "fscale": VARIABLE_INFO[VARIABLE]["scale"],
        "frequency": "daily",
        "nens": nens,
    }

data_dict = {
    name: {
        "seg1": segment(base_run, 2011, nens),
        "seg2": segment(forecast_run, 2012, nens),
    }
    for name, (base_run, forecast_run, nens) in MODEL_EXPERIMENTS.items()
}

# GPCP is available only for the precipitation configuration.
if VARIABLE == "PRECT":
    data_dict = {
        "GPCP": {
            "seg1": {
                "path": str(INPUT_DATA_ROOT / "Observations/GPCP/daily"),
                "template": "PRECT.daily.2011.nc",
                "period": "20111201-20111231",
                "fscale": 1.0,
                "frequency": "daily",
                "nens": 1,
            },
            "seg2": {
                "path": str(INPUT_DATA_ROOT / "Observations/GPCP/daily"),
                "template": "PRECT.daily.2012.nc",
                "period": "20120101-20120229",
                "fscale": 1.0,
                "frequency": "daily",
                "nens": 1,
            },
        },
        **data_dict,
    }


## Run diagnostics


In [ ]:
import cmocean

rmm_input_dir = INPUT_DATA_ROOT / "Observations/MJO_INDEX"
rmm_cache_dir = WORKFLOW_DATA_DIR / "rmm_index"
figure_mode = "mjo_filter" if MJO_FILTER else "remove_clim" if REMOVE_CLIMATOLOGY else "raw"
plot_settings = {
    "PRECT": {"vmin": -5, "vmax": 5, "nlevs": 21, "cmap": cmocean.cm.curl},
    "FLUT": {"vmin": -50, "vmax": 50, "nlevs": 51, "cmap": cmocean.cm.curl},
}[VARIABLE]

analyzer = HovmollerPrecipAnalyzer(
    data_dict=data_dict,
    varname=VARIABLE,
    var_info=VARIABLE_INFO[VARIABLE],
    lat_bounds=REGION_BOUNDS,
    dt_hours=24,
    mjo_filter=MJO_FILTER,
    remove_clim=REMOVE_CLIMATOLOGY,
    rmm_path=str(rmm_input_dir),
    rmm_cache_dir=str(rmm_cache_dir),
    frequency="daily",
    time_start=TIME_START,
    time_end=TIME_END,
    fontz=14,
)

analyzer.plot_rmm_phase_space(
    start=TIME_START,
    end=TIME_END,
    amp_thresh=1.0,
    savepath=str(WORKFLOW_FIGURE_DIR / "rmm_phase_space.pdf"),
)

for experiment in data_dict:
    print(f"\n[INFO] Running {VARIABLE} MJO diagnostics for {experiment}")
    experiment_figure_dir = WORKFLOW_FIGURE_DIR / experiment
    experiment_figure_dir.mkdir(parents=True, exist_ok=True)
    analyzer.generate_hovmoller(
        exp_key=experiment,
        savepath=str(experiment_figure_dir / f"hovmoller_{VARIABLE}_{figure_mode}.pdf"),
        filter_by_amplitude=True,
        amp_thresh=1.0,
        overlay_phase=True,
        cmap=plot_settings["cmap"],
        vmin=plot_settings["vmin"],
        vmax=plot_settings["vmax"],
        nlevs=plot_settings["nlevs"],
    )
    composites = analyzer.compute_phase_composites(experiment)
    composites.to_netcdf(WORKFLOW_DATA_DIR / f"{VARIABLE}_{experiment}_phase_composites.nc")
    analyzer.plot_phase_composites(
        composites,
        str(experiment_figure_dir / f"phase_composites_{VARIABLE}.pdf"),
    )
